## LIBRARY IMPORTS AND MODEL CONFIGURATIONS

In [ ]:
#Import all of the libraries that will be used for this project
from sklearn.metrics import accuracy_score,precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import train_test_split
from datasets import Dataset, load_dataset, DatasetDict, ClassLabel, concatenate_datasets
from scipy.special import softmax
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# HF = human-vs-AI-generated dataset from Hugging Face (ahmadreza13/human-vs-Ai-generated-dataset)
# CB = dataset from the M-DAIGT (Multi-Domain Detection of AI-Generated Text) shared task,
# hosted on CodaBench. Not redistributed here — see README for source and citation.

In [ ]:
#Calls on the transfomer model, and the tokeniser to be used
transformer_model = "distilroberta-base"
dataset_tk = AutoTokenizer.from_pretrained(transformer_model)

In [ ]:
#Tokenise function that converts texts into data the model can understand, add sentence padding and sets a max token length.
def tokenise(segment):
    return dataset_tk(
        segment["data"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
#Evaluation function that checks the models predictions against the true class labels, returning the peformance scores
def eval_scores(eval_pred):
    confidence_score, true_classes = eval_pred
    model_preds = np.argmax(confidence_score, axis=1)
    precision,recall,f1, _ = precision_recall_fscore_support( true_classes, model_preds, average="binary")
    accuracy = accuracy_score(true_classes, model_preds)
    return { "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}
    

In [ ]:
#TRAINING CONFIGS THAT TELLS THE TRAINER HOW THE MODEL SHOULD BE TRAINED.
#Training configurations for training on dataset HF
hf_training_conf = TrainingArguments(
    output_dir="./bert_results_hf",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=1,
    data_seed=1
)

In [ ]:
#Training configurations for training on dataset CB
cb_training_conf = TrainingArguments(
    output_dir="./bert_results_cb",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=1,
    data_seed=1,
)

In [ ]:
#Training configurations for training on dataset HF+CB
combo_training_conf = TrainingArguments(
    output_dir="./bert_results_combo",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=1,
    data_seed=1,
)

## LOADING AND CLEANING ALL THE DATASETS

In [ ]:
#Loading HF dataset, converting it to pandas for cleaning and filtering
hf_raw = load_dataset("ahmadreza13/human-vs-Ai-generated-dataset")
hf_df = hf_raw["train"].to_pandas()
#Splits the table into human/AI groups according to the Label
human_df = hf_df[hf_df["generated"] == 0]
ai_df    = hf_df[hf_df["generated"] == 1]

#Takes 10k sample from each group, ensuring near-equal group size. Fixed seed has been used to maintain reprodocuability
human_sample = human_df.sample(n=10000, random_state=1)
ai_sample    = ai_df.sample(n=10000, random_state=1)

#Cleans the dataset by removing duplicates in each group, ensuring reduction in data leakage
human_clean = human_sample.drop_duplicates(subset=["data"]).reset_index(drop=True)
ai_clean    = ai_sample.drop_duplicates(subset=["data"]).reset_index(drop=True)

#Joins the group of datas together, and de-duplicates it once more, incase there are any duplicates shared within the groups.
hf_balanced_df = pd.concat([human_clean, ai_clean], ignore_index=True)
hf_balanced_df = hf_balanced_df.drop_duplicates(subset=["data"]).reset_index(drop=True)

#Reshuffles the dataset, to ensure AI/Human text are in random order and not 00000s and 11111s. This reduces possibility of Model learning shortcuts
hf_balanced_df = hf_balanced_df.sample(frac=1, random_state=1).reset_index(drop=True)

#Convert back into Hugging face format
hf_balanced_ds = Dataset.from_pandas(hf_balanced_df, preserve_index=False)

In [ ]:
#Load CD dataset,and join them together 
cb_ds1 = pd.read_csv("task1-train.csv")
cb_ds2 = pd.read_csv("task2-train.csv")
cb_all_ds = pd.concat([cb_ds1, cb_ds2], ignore_index=True)

#Changes the label column to fit the models version of human/machine
cb_all_ds["label"] = cb_all_ds["label"].map({"human": 0, "machine": 1})

#Rename CB Column so it matches HF dataset columns, incase for future use
cb_all_ds = cb_all_ds.rename(columns={"text": "data","label": "generated"})

#Remove any duplicates that occur between the two CB datasets
cb_clean_df = cb_all_ds.drop_duplicates(subset=["data"]).reset_index(drop=True)

# CONVERT TO HF DATASET
cb_clean_ds = Dataset.from_pandas(cb_clean_df, preserve_index=False)

## TRAIN AND TEST SPLITS FOR EACH DATASET

In [ ]:
#HF DATASET TRAIN AND TEST SPLITS 
#Adds class labels to the 0s and 1s, as hugging face stratifcaiton expects a class label.
hf_balanced_ds = hf_balanced_ds.cast_column("generated", ClassLabel(names=["human", "ai"]))

#Create and store the first split of 80% train and 20% test split (columns are stratified so there is an equal relative split across sets)
split_hf_clean = hf_balanced_ds.train_test_split(test_size=0.2, seed=1, stratify_by_column="generated")
#Split the intial 80% train set into 90%/10% to get 72% train 8% validation 
second_split_hf = split_hf_clean["train"].train_test_split(test_size=0.1, seed=1, stratify_by_column="generated")

#Store the actual train/val/test as objects
hf_train = second_split_hf["train"] 
hf_val   = second_split_hf["test"]    
hf_test  = split_hf_clean["test"]  

#Put in a dictionary so it can all be tokenised at once
hf_ds = DatasetDict({
    "train": hf_train,
    "val": hf_val,
    "test": hf_test
})

#Tokenise all the text, rename HF's class column to "labels" for the trainers
hf_tokenised_ds = hf_ds.map(tokenise, batched=True)
hf_tokenised_ds = hf_tokenised_ds.rename_column("generated", "labels")
hf_tokenised_ds = hf_tokenised_ds.remove_columns(["data"])
hf_tokenised_ds.set_format("torch")


In [ ]:
#CB DATASET TRAIN AND TEST SPLITS
#Adds class labels to the 0s and 1s, as hugging face stratifcaiton expects a class label.
cb_clean_ds = cb_clean_ds.cast_column("generated", ClassLabel(names=["human", "ai"]))

#Create and store the first split of 80% train and 20% test split (columns are stratified so there is an equal relative split across sets)
split_cb_clean = cb_clean_ds.train_test_split(test_size=0.2, seed=1, stratify_by_column="generated")
#Split the intial 80% train set into 90%/10% to get 72% train 8% validation 
second_split_cb = split_cb_clean["train"].train_test_split(test_size=0.1, seed=1,  stratify_by_column="generated")

#Store the actual train/val/test as objects
cb_train = second_split_cb["train"]
cb_val   = second_split_cb["test"]
cb_test  = split_cb_clean["test"]

#Put in a dictionary so it can all be tokenised at once
cb_ds = DatasetDict({
    "train": cb_train,
    "val": cb_val,
    "test": cb_test
})

#Tokenise all the text, rename CB's class column to "labels" for the trainers
tokenised_cb_ds = cb_ds.map(tokenise, batched=True)
tokenised_cb_ds = tokenised_cb_ds.rename_column("generated", "labels")
tokenised_cb_ds = tokenised_cb_ds.remove_columns(["data"])
tokenised_cb_ds.set_format("torch")

In [ ]:
#Create the combined Dataset with HF+CB
#adds the hf and cb train and  tests together and shuffles it around
combo_train_full = concatenate_datasets([
    split_hf_clean["train"],
    split_cb_clean["train"]
]).shuffle(seed=1)
combo_test = concatenate_datasets([
    split_hf_clean["test"],
    split_cb_clean["test"]
]).shuffle(seed=1)


In [ ]:
#COMBO DATASET TRAIN AND TEST SPLIT
#Adds class labels to the 0s and 1s, as hugging face stratifcaiton expects a class label.
combo_train_full = combo_train_full.cast_column(
    "generated",
    ClassLabel(names=["human", "ai"])
)
combo_test = combo_test.cast_column(
    "generated",
    ClassLabel(names=["human", "ai"])
)

#split the #Split the intial 80% train set into 90%/10% to get 72% train 8% validation 
second_split_combo = combo_train_full.train_test_split(
    test_size=0.1,
    seed=1,
    stratify_by_column="generated"
)

#Put in a dictionary so it can all be tokenised at once
combo_ds = DatasetDict({
    "train": second_split_combo["train"],
    "val": second_split_combo["test"],
    "test": combo_test
})

#Tokenise all the text, rename Combo's class column to "labels" for the trainers
tokenised_combo = combo_ds.map(tokenise, batched=True)
tokenised_combo = tokenised_combo.rename_column("generated", "labels")
tokenised_combo = tokenised_combo.remove_columns(["data", "model", "id"])
tokenised_combo.set_format("torch")

## TRAINING FOR EACH DATASETS

In [ ]:
#HF----->HF #TRAINER MODEL FOR HF TO HF
hf_model = AutoModelForSequenceClassification.from_pretrained(transformer_model, num_labels=2)
hf_trainer = Trainer(
    model=hf_model,
    args=hf_training_conf,
    train_dataset=hf_tokenised_ds["train"],
    eval_dataset=hf_tokenised_ds["val"],
    compute_metrics=eval_scores,
)
hf_trainer.train()

In [ ]:
#cb---->cb TRAINER FOR CB TO CB
cb_model = AutoModelForSequenceClassification.from_pretrained(transformer_model, num_labels=2)
trainer_cb = Trainer(
    model=cb_model,
    args=cb_training_conf,
    train_dataset=tokenised_cb_ds["train"],
    eval_dataset=tokenised_cb_ds["val"],
    compute_metrics=eval_scores,
)
trainer_cb.train()

In [ ]:
combo_model = AutoModelForSequenceClassification.from_pretrained(transformer_model, num_labels=2)
trainer_combo = Trainer(
    model=combo_model,
    args=combo_training_conf,
    train_dataset=tokenised_combo["train"],
    eval_dataset=tokenised_combo["val"],
    compute_metrics=eval_scores,
)
trainer_combo.train()

## TESTING ON EACH IN-DOMAIN, CROSS-DOMAIN AND COMBINED DOMAIN DATASET SETTINGS 

In [ ]:
#HF---->HF #EVALUATOR FOR HF TO HF 
hf_pred_output = hf_trainer.predict(hf_tokenised_ds["test"])
model_prediction = np.argmax(hf_pred_output.predictions, axis=1)
correct_class = hf_pred_output.label_ids
accuracy = accuracy_score(correct_class, model_prediction)
precision, recall, f1, _ = precision_recall_fscore_support(correct_class, model_prediction, average="binary")
cm = confusion_matrix(correct_class, model_prediction)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Confusion Matrix:")
print(cm)

In [ ]:
#HF---->CB #EVALUATOR FOR HF TO CB
pred_output_hf_to_cb = hf_trainer.predict(tokenised_cb_ds["test"])
model_prediction = np.argmax(pred_output_hf_to_cb.predictions, axis=1)
correct_class = pred_output_hf_to_cb.label_ids
accuracy = accuracy_score(correct_class, model_prediction)
precision, recall, f1, _ = precision_recall_fscore_support(correct_class, model_prediction, average="binary")
cm = confusion_matrix(correct_class, model_prediction)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Confusion Matrix:")
print(cm)

In [ ]:
#CB---->CB #EVALUATOR FOR CB TO CB
pred_output_cb = trainer_cb.predict(tokenised_cb_ds["test"])
model_prediction = np.argmax(pred_output_cb.predictions, axis=1)
correct_class = pred_output_cb.label_ids
accuracy = accuracy_score(correct_class, model_prediction)
precision, recall, f1, _ = precision_recall_fscore_support(correct_class, model_prediction, average="binary")
cm = confusion_matrix(correct_class, model_prediction)


print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Confusion Matrix:")
print(cm)

In [ ]:
#CB--------> HF #CB TO HF EVALUATOR
pred_output_cb_to_hf = trainer_cb.predict(hf_tokenised_ds["test"])
model_prediction = np.argmax(pred_output_cb_to_hf.predictions, axis=1)
correct_class = pred_output_cb_to_hf.label_ids
accuracy = accuracy_score(correct_class, model_prediction)
precision, recall, f1, _ = precision_recall_fscore_support(correct_class, model_prediction, average="binary")
cm = confusion_matrix(correct_class, model_prediction)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Confusion Matrix:")
print(cm)

In [ ]:
#COMBO--------> COMBO #COMBO TO COMBO EVALUATOR
pred_output_combo = trainer_combo.predict(tokenised_combo["test"])
model_prediction = np.argmax(pred_output_combo.predictions, axis=1)
correct_class = pred_output_combo.label_ids
accuracy = accuracy_score(correct_class, model_prediction)
precision, recall, f1, _ = precision_recall_fscore_support(correct_class, model_prediction, average="binary")
cm = confusion_matrix(correct_class, model_prediction)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Confusion Matrix:")
print(cm)

In [ ]:
#COMBO--------> HF #COMBO TO HF EVALUATOR
pred_output_combo_to_hf = trainer_combo.predict(hf_tokenised_ds["test"])
model_prediction = np.argmax(pred_output_combo_to_hf.predictions, axis=1)
correct_class = pred_output_combo_to_hf.label_ids
accuracy = accuracy_score(correct_class, model_prediction)
precision, recall, f1, _ = precision_recall_fscore_support(correct_class, model_prediction, average="binary")
cm = confusion_matrix(correct_class, model_prediction)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Confusion Matrix:")
print(cm)

In [ ]:
#COMBO--------> CB #COMBO TO CB EVALUATOR
pred_output_combo_to_cb = trainer_combo.predict(tokenised_cb_ds["test"])
model_prediction = np.argmax(pred_output_combo_to_cb.predictions, axis=1)
correct_class = pred_output_combo_to_cb.label_ids
accuracy = accuracy_score(correct_class, model_prediction)
precision, recall, f1, _ = precision_recall_fscore_support(correct_class, model_prediction, average="binary")
cm = confusion_matrix(correct_class, model_prediction)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("Confusion Matrix:")
print(cm)

## EXTENDED COMBINED DATASET THRESHOLD TUNING AND LEARNING CURVE EXPERIMENT 

In [ ]:
#THRESHOLD TUNING EXPERIMENT 

#Turns the combo settings predictions and true classes, turning the predictions into percentages, taking only AI percentages
combo_correct_class = pred_output_combo.label_ids
combo_models_predictions = pred_output_combo.predictions
probs_combo = softmax(combo_models_predictions, axis=1)[:, 1]

#creates threshold ranges from 0.1-0.9
thresholds = np.arange(0.1, 1.01, 0.2)
results = []

#checks if predicted AI percentages are greater than the set threshold percentage
for threshold in thresholds:
    threshold_predictions = (probs_combo >= threshold).astype(int)

    #compares against the real answers 
    threshold_precision, threshold_recall, threshold_f1, _ = precision_recall_fscore_support(
    combo_correct_class,
    threshold_predictions,
    average="binary"
    )
    threshold_accuracy = accuracy_score(combo_correct_class, threshold_predictions)
    threshold_cm = confusion_matrix(combo_correct_class, threshold_predictions)


    #appends the results into the list
    results.append({
        "threshold": round(float(threshold), 2),
        "accuracy": threshold_accuracy,
        "precision": threshold_precision,
        "recall": threshold_recall,
        "f1": threshold_f1
    })

    #prints the outputs in better fashion
    print("Threshold:", threshold)
    print("Accuracy :", threshold_accuracy)
    print("Precision:", threshold_precision)
    print("Recall   :", threshold_recall)
    print("F1       :", threshold_f1)
    print("Confusion Matrix:")
    print(threshold_cm)
    print()

In [ ]:
#LEARNING CURVE EXPERIEMENT
training_fractions = [0.1, 0.25, 0.5, 1.0]
learning_curve_results = []

#Creates a model training loop for all the training set size fractions
for fraction in training_fractions:
    
    #The model resets for every new training fraction 
    learning_curve_model = AutoModelForSequenceClassification.from_pretrained(
        transformer_model,
        num_labels=2
    )
    
    #Creates learning curve configuration for the model for every new training fraction
    learning_curve_configs = TrainingArguments(
        output_dir=f"./learning_curve_combo_{int(fraction*100)}",
        num_train_epochs=2,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        report_to="none",
        seed=1,
        data_seed=1,
    )

    #Creates a fraction training size for the set fraction run from the original combo training set
    fraction_train_size = combo_ds["train"].shuffle(seed=1).select(range(int(len(combo_ds["train"]) * fraction)))

    #tokenise all fraction training size, validation set and test set again.
    fraction_train_tokeniser = fraction_train_size.map(tokenise, batched=True)
    combo_val_tokeniser = combo_ds["val"].map(tokenise, batched=True)
    combo_test_tokeniser = combo_ds["test"].map(tokenise, batched=True)

    #Renaming all the columns again for good code practice  and the hugging face trainer 
    fraction_train_tokeniser = fraction_train_tokeniser.rename_column("generated", "labels")
    combo_val_tokeniser = combo_val_tokeniser.rename_column("generated", "labels")
    combo_test_tokeniser = combo_test_tokeniser.rename_column("generated", "labels")

    #Dropping all the uneeded columns again for good code practice
    fraction_train_tokeniser = fraction_train_tokeniser.remove_columns(["data", "model", "id"])
    combo_val_tokeniser = combo_val_tokeniser.remove_columns(["data", "model", "id"])
    combo_test_tokeniser = combo_test_tokeniser.remove_columns(["data", "model", "id"])

    #converts data to pytorch tensors for the model
    fraction_train_tokeniser.set_format("torch")
    combo_val_tokeniser.set_format("torch")
    combo_test_tokeniser.set_format("torch")

    #Learning curve model trainer and evaluator
    learning_curve_trainer = Trainer(
            model=learning_curve_model,
            args=learning_curve_configs,
            train_dataset=fraction_train_tokeniser,
            eval_dataset=combo_val_tokeniser,
            compute_metrics=eval_scores,
    )
        
    learning_curve_trainer.train()
    learning_curve_predictions = learning_curve_trainer.predict(combo_test_tokeniser)

    #Learning curve results
    model_prediction = np.argmax(learning_curve_predictions.predictions, axis=1)
    correct_class = learning_curve_predictions.label_ids
    precision, recall, f1, _ = precision_recall_fscore_support(correct_class, model_prediction, average="binary")
    accuracy = accuracy_score(correct_class, model_prediction)

    learning_curve_results.append({
        "train_percent": int(fraction * 100),
        "train_size": len(fraction_train_tokeniser),
        "accuracy": accuracy,
        "f1": f1
    })

learning_curve_df = pd.DataFrame(learning_curve_results)
learning_curve_df